# Olist E-commerce: Delivery Performance and Customer Reviews

**Questions:** How long does delivery take and how often is it late?
Do late deliveries get worse reviews? Where and when are deliveries late?

**Late** = delivered after the estimated delivery date shown to the customer.

In [1]:
import duckdb

%load_ext sql
con = duckdb.connect("../data/olist.duckdb", read_only=True)
%sql con
%config SqlMagic.displaylimit = 50

In [2]:
%%sql
SELECT reviews_per_order, COUNT(*) AS orders
FROM (
    SELECT order_id, COUNT(*) AS reviews_per_order
    FROM reviews
    GROUP BY order_id
)
GROUP BY reviews_per_order
ORDER BY reviews_per_order;

Running query in 'DuckDBPyConnection'

reviews_per_order,orders
1,98126
2,543
3,4


In [3]:
%%sql
SELECT review_score,
       COUNT(*) AS reviews,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM reviews
GROUP BY review_score
ORDER BY review_score;

Running query in 'DuckDBPyConnection'

review_score,reviews,pct
1,11424,11.5
2,3151,3.2
3,8179,8.2
4,19142,19.3
5,57328,57.8


In [4]:
%%sql --save deliveries --no-execute
WITH latest_review AS (
    SELECT order_id, review_score
    FROM (
        SELECT order_id,
               review_score,
               ROW_NUMBER() OVER (PARTITION BY order_id
                                  ORDER BY review_answer_timestamp DESC, review_id) AS rn
        FROM reviews
    )
    WHERE rn = 1
)
SELECT o.order_id,
       c.customer_state,
       date_trunc('month', o.order_purchase_timestamp)::DATE AS month,
       date_diff('day', o.order_purchase_timestamp, o.order_delivered_customer_date) AS delivery_days,
       date_diff('day', o.order_estimated_delivery_date, o.order_delivered_customer_date) AS days_late,
       r.review_score
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN latest_review r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'

Running query in 'DuckDBPyConnection'

Skipping execution...

In [5]:
%%sql
SELECT COUNT(*) AS orders,
       COUNT(review_score) AS orders_with_review,
       ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
       MEDIAN(delivery_days) AS median_delivery_days,
       ROUND(AVG(days_late), 1) AS avg_days_vs_estimate,
       ROUND(100.0 * AVG(CASE WHEN days_late > 0 THEN 1 ELSE 0 END), 1) AS late_pct,
       ROUND(AVG(review_score), 2) AS avg_review
FROM deliveries;

Generating CTE with stored snippets: 'deliveries'

Running query in 'DuckDBPyConnection'

orders,orders_with_review,avg_delivery_days,median_delivery_days,avg_days_vs_estimate,late_pct,avg_review
96203,95560,12.5,10.0,-11.8,6.8,4.16


In [6]:
%%sql
SELECT CASE WHEN days_late > 0 THEN 'Late' ELSE 'On time' END AS delivery_status,
       COUNT(*) AS orders,
       ROUND(AVG(review_score), 2) AS avg_review,
       ROUND(100.0 * AVG(CASE WHEN review_score = 1 THEN 1 ELSE 0 END), 1) AS pct_1_star,
       ROUND(100.0 * AVG(CASE WHEN review_score = 5 THEN 1 ELSE 0 END), 1) AS pct_5_star
FROM deliveries
WHERE review_score IS NOT NULL
GROUP BY delivery_status
ORDER BY delivery_status DESC;

Generating CTE with stored snippets: 'deliveries'

Running query in 'DuckDBPyConnection'

delivery_status,orders,avg_review,pct_1_star,pct_5_star
On time,89182,4.29,6.6,62.3
Late,6378,2.27,53.8,16.5


In [7]:
%%sql
SELECT CASE WHEN days_late <= -10 THEN '1. 10+ days early'
            WHEN days_late <= 0   THEN '2. 0-9 days early'
            WHEN days_late <= 3   THEN '3. 1-3 days late'
            WHEN days_late <= 7   THEN '4. 4-7 days late'
            WHEN days_late <= 14  THEN '5. 8-14 days late'
            ELSE '6. 15+ days late'
       END AS delivery_group,
       COUNT(*) AS orders,
       ROUND(AVG(review_score), 2) AS avg_review,
       ROUND(100.0 * AVG(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END), 1) AS pct_bad_review
FROM deliveries
WHERE review_score IS NOT NULL
GROUP BY delivery_group
ORDER BY delivery_group;

Generating CTE with stored snippets: 'deliveries'

Running query in 'DuckDBPyConnection'

delivery_group,orders,avg_review,pct_bad_review
1. 10+ days early,61270,4.32,8.9
2. 0-9 days early,27912,4.22,10.1
3. 1-3 days late,1851,3.29,32.1
4. 4-7 days late,1748,2.1,67.7
5. 8-14 days late,1446,1.67,80.2
6. 15+ days late,1333,1.72,78.3


In [8]:
%%sql
SELECT customer_state,
       COUNT(*) AS orders,
       ROUND(100.0 * AVG(CASE WHEN days_late > 0 THEN 1 ELSE 0 END), 1) AS late_pct,
       ROUND(AVG(delivery_days), 1) AS avg_delivery_days,
       ROUND(AVG(review_score), 2) AS avg_review
FROM deliveries
GROUP BY customer_state
HAVING COUNT(*) >= 500
ORDER BY late_pct DESC;

Generating CTE with stored snippets: 'deliveries'

Running query in 'DuckDBPyConnection'

customer_state,orders,late_pct,avg_delivery_days,avg_review
MA,713,17.5,21.5,3.83
CE,1273,13.8,21.2,3.94
BA,3253,12.2,19.3,3.93
RJ,12310,12.1,15.2,3.97
PA,942,11.3,23.7,3.91
ES,1992,10.7,15.7,4.08
PB,516,10.5,20.4,4.07
MS,701,9.7,15.5,4.16
PE,1587,9.6,18.4,4.08
SC,3537,8.2,14.9,4.13


In [9]:
%%sql
SELECT month,
       COUNT(*) AS orders,
       ROUND(100.0 * AVG(CASE WHEN days_late > 0 THEN 1 ELSE 0 END), 1) AS late_pct,
       ROUND(AVG(review_score), 2) AS avg_review
FROM deliveries
GROUP BY month
ORDER BY month;

Generating CTE with stored snippets: 'deliveries'

Running query in 'DuckDBPyConnection'

month,orders,late_pct,avg_review
2017-01-01,750,2.9,4.2
2017-02-01,1653,3.0,4.2
2017-03-01,2546,4.6,4.19
2017-04-01,2303,6.6,4.14
2017-05-01,3545,3.0,4.24
2017-06-01,3135,3.0,4.22
2017-07-01,3872,2.8,4.26
2017-08-01,4193,2.9,4.31
2017-09-01,4150,4.4,4.27
2017-10-01,4478,4.2,4.2


In [10]:
%%sql
SELECT ROUND(CORR(days_late, review_score), 3) AS corr_days_late_vs_review,
       ROUND(CORR(delivery_days, review_score), 3) AS corr_delivery_days_vs_review
FROM deliveries;

Generating CTE with stored snippets: 'deliveries'

Running query in 'DuckDBPyConnection'

corr_days_late_vs_review,corr_delivery_days_vs_review
-0.27,-0.334


In [11]:
con.close()

## Findings: delivery and reviews

- **Delivery:** median 10 days (average 12.5). Olist's estimates are generous: orders arrive
  ~12 days before the promised date on average, yet **6.8% are still late**.
- **Late orders destroy reviews:** average **2.27★ vs 4.29★** on time; **53.8% of late orders get
  1 star** vs 6.6% on time (8× more).
- **Dose-response:** 1–3 days late already triples bad reviews (10% → 32%); after a week late,
  ~80% of reviews are bad. Very early deliveries score best: customers judge against the promise.
- **Where:** the Northeast (MA 17.5%, CE 13.8%, BA 12.2%) and **Rio de Janeiro (12.1%, Olist's
  #2 market)** are late 3–4× more often than São Paulo (4.5%).
- **When:** Black Friday Nov 2017 (12.4% late) and **Feb–Mar 2018 (14–19% late)**, the months
  with the lowest reviews.
- **Correlation:** −0.27 (days late vs review). It's modest as a single number because most orders are
  early; the damage is concentrated in late orders.

**Limitation:** this is observational data, so it shows a strong association, not proof. The
dose-response pattern and the obvious mechanism make causation likely, but an experiment would
confirm it.

## Recommendations
1. **Fix logistics in Rio and the Northeast first**: the most late orders and the biggest review damage.
2. **Plan carrier capacity before Black Friday** and investigate what went wrong in Feb–Mar 2018.
3. **Keep estimates realistic but safe**: early delivery delights; a missed promise is punished hard.
4. **Proactive apology for late orders** (message + small voucher) before the customer leaves a review.

## SQL lessons
- Reviews had duplicates per order → `ROW_NUMBER()` + tie-breaker to keep one.
- **Reusable query:** a `TEMP VIEW` didn't work because JupySQL runs each query on its own
  connection. JupySQL's saved snippet (`%%sql --save name --no-execute`) stores the query and
  inserts it as a CTE wherever it's used.
- `IS NULL`, not `= NULL`. `AVG` skips NULLs. `HAVING` filters groups; `WHERE` filters rows.